In [3]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List

In [ ]:
@dataclass
class PetProfile:
    pet_id: str
    pet_name: Optional[str] = None
    breed: str                      # for searching any special attribute of the breed
    birth_date: datetime
    is_high_energy: bool            # e.g., Vizsla 
    postcode: str                   # For seasonal/geographic tick risk 
    streak: int                     # auto calculate number of checkin in (consecutive), reset if hasn't checkin more than 1 week (7 days)
    consistency: float               # Automatic check if has owner has checkin regularly
    worming_compliant: bool         # Non-compliance moves score to Action Needed 
    vaccination_current: bool       # Treatment currency feeds score 
    days_since_last_vet_visit: int  # Auto-calculated metric 

@dataclass
class VetInteractionLog:
    """Vet log after checkin"""
    visited_vet: bool
    reason_for_visit: Optional[str] = None # Categorized reason informs score 
    log: str = None

@dataclass
class DentalHealthLog:
    """Periodic check-in (Puppy teething, 6-month baseline, and 6-monthly ongoing)"""
    breath_odour: str             # e.g., "Noticeably worse", "Normal" 
    chewing_changes: str          # e.g., "Less interested", "Favouring one side" 
    gum_redness_bleeding: str     # e.g., "Noticed buildup/redness", "Normal" 
    professional_clean_due: bool  # True/False
    
@dataclass
class ToiletingLog:
    """Tracks bowel habits, frequency, and acute overrides"""
    frequency_changes: str    # e.g., "Normal", "More frequent", "Less frequent"
    consistency_changes: str  # e.g., "Normal", "Runny", "Hard" 
    indoor_accidents: bool    # True/False 
    blood_present: bool


@dataclass
class CheckInData:
    """The Weekly Check-in Data payload"""
    pet_id: str
    timestamp: datetime
    
    # Pre-Moment 1 Context
    vet_interaction: VetInteractionLog 
    
    # Core Weekly Vitals (Moment 1 & 2)
    energy: int                  # 1-10 scale (comparative to baseline) 
    appetite: int                # 1-10 scale 
    sleep_quality: int           # 1-10 scale 
    water_intake: str            # "Normal", "Increased", "Decreased" (Increased = High Risk) 
    activity_minutes: int        # Aggregated from Strava-style quick-logs 
    
    # Behaviour & Changes
    diet_changed: bool           # Elevated if coincides with health signal 
    social_behaviour: str        # e.g., "Normal", "Withdrawn", "Clingy", "Aggressive" 
    
    # Physical & Specific Logs
    physical_health: Dict[str, int] # e.g., {"eyes": 10, "ears": 8, "coat": 9} 
    toileting: ToiletingLog         # Modular sub-class 
    
    # Parasite & Pain Checks (Moment 1 Additions)
    tick_check_completed: bool      # Passive score contributor if consistently done 
    skin_or_lump_issue_found: bool  # True/False 
    is_in_pain: bool                # Triggers quarterly or from combined flags 
    
    # Periodic & Monthly Inputs (Can be None/Null on standard weeks)
    weight: Optional[float] = None              # First check-in of each month [cite: 310]
    dental_log: Optional[DentalHealthLog] = None # Periodic (weeks 12-24, or 6-monthly) [cite: 213]
    
    # Closing Question (Breeder / Owner Support)
    owner_wellbeing_status: str     # Flags breeder dashboard if struggling 2+ weeks 

class OutputData:
    """Output"""
    vitality_score: Optional[int]
    zone: str
    trend: str
    confidence: str
    top_drivers: List[str]
    red_flag: bool
    recommended_next_step: str
    owner_summary: str
    

For now weight are indicated manually, weighting may changes base on pet age, breed,...

In [ ]:
class VitalityScoreEngine:
    """
    Layer 1: Calculates the 0-100 wellness score based on comparative baseline logic and dynamic breed weighting.
    """
    def __init__(self, pet: PetProfile, history: list[CheckInData]):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline (should use only 4 previous week)

    def calculate_score(self, current: CheckInData) -> OutputData:
        
        # Acute Overrides
        is_sudden_collapse = (current.energy <= 2 and current.appetite <= 2)

        if current.toileting.blood_present or current.water_intake == "Increased" or is_sudden_collapse:
            return self._generate_action_needed_override(current)
        
        # Checking if meet the requirement length of the length is met
        if len(self.history) < 4:
            return OutputData(
                vitality_score=None,
                zone="Building Baseline",
                trend="N/A",
                confidence="Low",
                top_drivers=["Establishing normal patterns"],
                red_flag=False,
                recommended_next_step="Continue completing weekly check-ins.",
                owner_summary="Evie is still learning what is normal for your pet during these first four weeks."
            )
        
        # Average history baseline
        avg_energy = sum(h.energy for h in self.history) / 4.0
        avg_appetite = sum(h.appetite for h in self.history) / 4.0
        avg_sleep = sum(h.sleep_quality for h in self.history) / 4.0
        avg_act = max(1, sum(h.activity_minutes for h in self.history) / 4.0)

        # Calculate Negative Deviations
        drop_energy = max(0, avg_energy - current.energy)                 # Energy
        drop_appetite = max(0, avg_appetite - current.appetite)           # Appetite
        drop_sleep = max(0, avg_sleep - current.sleep_quality)            # Sleep
        drop_act = max(0, (avg_act - current.activity_minutes) / avg_act) # Activity minute

        # Weighting (Normal breed v.s High Energy breed)
        w_act, w_sleep, w_energy, w_appetite = 0.25, 0.25, 0.25, 0.25
        if self.pet.is_high_energy:
            w_act = 0.40
            w_sleep = 0.10

        raw_score = 100.0
        drivers = []

        # Sighal v.s Noise
        if drop_act >= 1.0 and (drop_energy >= 1.0 or drop_sleep >= 1.0):
            raw_score -= 25.0
            drivers.append("Combined drop in appetite and energy/sleep")
        else:
            raw_score -= (drop_appetite * 10 * w_appetite)
            raw_score -= (drop_energy * 10 * w_energy)
            raw_score -= (drop_sleep * 10 * w_sleep)
            if drop_act  >= 1: drivers.append("Slightly lower appetite")
            if drop_energy >= 1: drivers.append("Slightly lower energy")

        # Activity Penalty
        if drop_act > 0.15:
            raw_score -= (drop_act * 30 * w_act)
            drivers.append("Lower activity than usual")

        # Medium weight vitals, platform data
        if current.social_behaviour in ["Withdrawn", "Aggressive", "Clingy"]:
            raw_score -= 10.0
            drivers.append(f"Social behaviour shift ({current.social_behaviour})")
            
        if not self.pet.worming_compliant:
            raw_score -= 10.0
            drivers.append("Overdue for worming treatment")

        final_score = max(0, min(100, int(raw_score)))
        zone_name = self._get_zone(final_score)
        
        if not drivers:
            drivers = ["Vitals are stable and consistent"]

        return OutputData(
            vitality_score=final_score,
            zone=zone_name,
            trend="Stable" if final_score >= 85 else "Slightly down", 
            confidence=self.pet.consistency,
            top_drivers=drivers,
            red_flag=False,
            recommended_next_step=self._get_next_step(zone_name),
            owner_summary=self._generate_summary(zone_name, drivers)
        )
    
    def _get_zone(self, score: int) -> str:
        if score >= 85: return "Bright Green"
        if score >= 70: return "Medium Green"
        if score >= 50: return "Watch Zone"
        return "Action Needed"

    def _get_next_step(self, zone: str) -> str:
        if zone in ["Bright Green", "Medium Green"]: return "Keep up the great routine."
        if zone == "Watch Zone": return "Monitor closely; consider reviewing our vet directory."
        return "Contact your veterinarian for advice." 

    def _generate_summary(self, zone: str, drivers: List[str]) -> str:
        if zone == "Bright Green":
            return "Your pet is doing wonderfully and tracking perfectly with their normal baseline."
        return f"Your pet seems a little below their usual pattern this week, mainly due to {drivers[0].lower()}."

        
        
        

In [ ]:
class RedFlagEngine:
    def evaluate(self, current: CheckInData) -> Optional[OutputData]:
        """
        Scans for acute overrides. Returns a critical OutputData object if a 
        red flag is found, otherwise returns None to allow Layer 1 to calculate normally.
        """
        acute_driver = None

        # Toileting Red Flags (Immediate bypass)
        if current.toileting.blood_present:
            acute_driver = "Blood present in stool"

        # Increased Water Intake (Chronic disease flag)
        elif current.water_intake == "Increased":
            acute_driver = "Significantly increased water intake"

        # Sudden Collapse (Combined lethargy and anorexia)
        # Using <= 2 on a 1-10 scale to represent "collapse"
        elif current.energy <= 2 and current.appetite <= 2:
            acute_driver = "Sudden collapse in appetite combined with lethargy"

        # Confirmed Tick / Parasite 
        elif current.skin_or_lump_issue_found and current.tick_check_completed:
            # Assuming the issue found during a tick check is a confirmed tick
            acute_driver = "Confirmed tick or parasite"

        # If any override was triggered, return the Action Needed object immediately
        if acute_driver:
            return OutputData(
                vitality_score=40,  # Forces the UI into the Below 50 zone
                zone="Action Needed",
                trend="Acute Drop",
                confidence="High",  # Acute signs are high confidence triggers
                top_drivers=[acute_driver],
                red_flag=True,
                recommended_next_step="Contact your veterinarian immediately.",
                owner_summary=f"We noticed {acute_driver.lower()}, which requires professional advice. Please contact your vet."
            )

        # No red flags detected; return None so the pipeline knows to proceed to Layer 1
        return None
    

In [ ]:
class DataPipelineOrchestrator:
    """
    Manages the flow of data through the PawHealthAI Vitality Score system.
    """
    def __init__(self, database_client):
        # This represents your database connection (e.g., PostgreSQL, Firebase)
        self.db = database_client 

    def process_new_checkin(self, current_checkin: CheckInData) -> OutputData:
        """
        The main pipeline executed when a user submits a weekly check-in.
        """
        # 1. Fetch current profile and historical baseline
        pet = self.db.get_pet_profile(current_checkin.pet_id)
        history = self.db.get_history(current_checkin.pet_id) # Retrieves last 4 weeks

        # 2. Layer 2: The Red Flag Engine (Safety Net First)
        red_flag_engine = RedFlagEngine()
        acute_result = red_flag_engine.evaluate(current_checkin)
        
        if acute_result:
            # If a red flag is triggered, completely bypass Layer 1 math
            return acute_result

        # 3. Layer 1: The Wellness Trend Engine (The Math)
        trend_engine = VitalityScoreEngine(pet, history)
        final_result = trend_engine.calculate_score(current_checkin)

        # 4. Save the new check-in to history
        # CRITICAL: This must happen AFTER calculation so we don't dilute the baseline
        self.db.save_checkin(current_checkin)

        return final_result

In [ ]:
"""Final product, assuming all the neccessary function has been built, required API for testing"""

def process_weekly_checkin(pet: PetProfile, current_checkin: CheckInData):
    red_flag_engine = RedFlagEngine()
    output = red_flag_engine.evaluate(current_checkin)
    
    if not output:
        trend_engine = VitalityScoreEngine(pet, db.get_history(pet.pet_id))
        output = trend_engine.calculate_score(current_checkin)
        
    db.save_checkin(current_checkin)
    
    return output